Notebook ini membandingkan CNN baseline, VGG16 baseline, dan ResNet50-U-Net yang sudah dilatih. Jalankan dari folder `backend/model`; `model_sdss.h5` harus tersedia di folder tersebut. Ketiga model dievaluasi pada test set yang sama.


In [1]:
import os, glob
import numpy as np
import cv2
import tensorflow as tf
# WSL GPU has limited available VRAM; grow the allocator as needed instead of
# reserving most of the GPU memory at runtime startup. Must run before GPU ops.
for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Conv2D, Conv2DTranspose, MaxPooling2D,
    BatchNormalization, Dropout, concatenate, Lambda)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
# TensorFlow 2.10 cannot use native CUDA kernels on compute capability 12.0 GPUs
# (e.g. RTX 50 series); hide that incompatible GPU to avoid a very long PTX JIT attempt.
_tf_version = tuple(int(part) for part in tf.__version__.split(".")[:2])
for _gpu in tf.config.list_physical_devices("GPU"):
    _details = tf.config.experimental.get_device_details(_gpu)
    _capability = _details.get("compute_capability")
    if _capability and _capability[0] >= 12 and _tf_version <= (2, 10):
        tf.config.set_visible_devices([], "GPU")
        print("TensorFlow", tf.__version__, "tidak kompatibel dengan GPU compute capability", _capability,
              "pada runtime ini; memakai CPU agar tidak macet saat kompilasi CUDA.")
        break
TRAINING_DIR = os.path.join("..", "..", "training")
IMAGES_DIR = os.path.join(TRAINING_DIR, "images")
TARGETS_DIR = os.path.join(TRAINING_DIR, "targets")
IMG_SIZE = (256, 256)
BATCH_SIZE = 2
EPOCHS = 15
LEARNING_RATE = 1e-4
OUTPUT_DIR = "comparison_baselines"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Training images:", os.path.abspath(IMAGES_DIR))

c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow 2.10.0 tidak kompatibel dengan GPU compute capability (12, 0) pada runtime ini; memakai CPU agar tidak macet saat kompilasi CUDA.
TensorFlow: 2.10.0
Training images: c:\Users\Jase.LAPTOP-UM736EL9\Documents\Binus\Thesis S2\SDSS_Post-Disaster_Food_Logistics\training\images


In [2]:
post_images = sorted(glob.glob(os.path.join(IMAGES_DIR, "*_post_disaster.png")))
image_pairs, target_paths = [], []
for post_path in post_images:
    pre_path = post_path.replace("_post_disaster.png", "_pre_disaster.png")
    scene_id = os.path.basename(post_path).replace("_post_disaster.png", "")
    mask_path = os.path.join(TARGETS_DIR, f"{scene_id}_post_disaster_target.png")
    if not os.path.exists(mask_path):
        continue
    if not os.path.exists(pre_path):
        pre_path = post_path
    image_pairs.append((pre_path, post_path))
    target_paths.append(mask_path)
if not image_pairs:
    raise FileNotFoundError(f"Tidak ada pasangan citra dan mask. Periksa: {IMAGES_DIR} dan {TARGETS_DIR}")

train_pairs, temp_pairs, train_masks, temp_masks = train_test_split(
    image_pairs, target_paths, test_size=0.30, random_state=SEED)
val_pairs, test_pairs, val_masks, test_masks = train_test_split(
    temp_pairs, temp_masks, test_size=0.50, random_state=SEED)
print(f"Total {len(image_pairs)} | train {len(train_pairs)} | val {len(val_pairs)} | test {len(test_pairs)}")

Total 2799 | train 1959 | val 420 | test 420


In [3]:
def load_pair_mask(pre_path, post_path, mask_path):
    # tf.py_function passes scalar EagerTensors; decode them to filesystem paths
    # before calling OpenCV (cv2.imread requires str/bytes, not EagerTensor).
    def decode_path(path):
        if hasattr(path, "numpy"):
            path = path.numpy()
        if isinstance(path, bytes):
            return path.decode("utf-8")
        if isinstance(path, np.ndarray):
            path = path.item()
            if isinstance(path, bytes):
                return path.decode("utf-8")
        return os.fsdecode(path)

    pre_path = decode_path(pre_path)
    post_path = decode_path(post_path)
    mask_path = decode_path(mask_path)

    def read_rgb(path):
        image = cv2.imread(path)
        if image is None:
            raise ValueError(f"Gagal membaca citra: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return cv2.resize(image, IMG_SIZE).astype(np.float32) / 255.0
    pre = read_rgb(pre_path)
    post = read_rgb(post_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Gagal membaca mask: {mask_path}")
    mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    mask = (mask > 0).astype(np.float32)[..., None]
    return np.concatenate([pre, post], axis=-1), mask

def tf_load(pre_path, post_path, mask_path):
    image, mask = tf.py_function(load_pair_mask, [pre_path, post_path, mask_path], [tf.float32, tf.float32])
    image.set_shape([*IMG_SIZE, 6])
    mask.set_shape([*IMG_SIZE, 1])
    return image, mask

def augment(image, mask):
    flip_lr = tf.random.uniform(()) > 0.5
    image = tf.cond(flip_lr, lambda: tf.image.flip_left_right(image), lambda: image)
    mask = tf.cond(flip_lr, lambda: tf.image.flip_left_right(mask), lambda: mask)
    flip_ud = tf.random.uniform(()) > 0.5
    image = tf.cond(flip_ud, lambda: tf.image.flip_up_down(image), lambda: image)
    mask = tf.cond(flip_ud, lambda: tf.image.flip_up_down(mask), lambda: mask)
    return image, mask

def make_dataset(pairs, masks, training=False):
    pre = [p[0] for p in pairs]
    post = [p[1] for p in pairs]
    ds = tf.data.Dataset.from_tensor_slices((pre, post, masks))
    if training:
        ds = ds.shuffle(len(pairs), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_pairs, train_masks, training=True)
val_ds = make_dataset(val_pairs, val_masks)
test_ds = make_dataset(test_pairs, test_masks)

In [4]:
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def bce_dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + (1.0 - dice_coef(y_true, y_pred))

def conv_block(x, filters, dropout=0.0):
    x = Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    x = Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = BatchNormalization()(x)
    if dropout:
        x = Dropout(dropout)(x)
    return x

def decoder(x, skips):
    for filters, skip in zip([256, 128, 64, 32], reversed(skips)):
        x = Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
        x = concatenate([x, skip])
        x = conv_block(x, filters, dropout=0.15)
    x = Conv2DTranspose(32, 2, strides=2, padding="same")(x)
    x = conv_block(x, 16)
    return Conv2D(1, 1, activation="sigmoid", name="damage_mask")(x)

def build_cnn_baseline():
    inputs = Input((*IMG_SIZE, 6), name="pre_post_rgb")
    x = inputs
    skips = []
    # Skip features at 128, 64, 32, 16 px, matching the VGG16 decoder.
    for filters in [16, 32, 64, 128]:
        x = conv_block(x, filters)
        x = MaxPooling2D(pool_size=2)(x)
        skips.append(x)
    x = conv_block(x, 256, dropout=0.2)  # 16x16
    x = MaxPooling2D(pool_size=2)(x)      # 8x8 bridge
    x = conv_block(x, 256, dropout=0.2)
    outputs = decoder(x, skips)
    return Model(inputs, outputs, name="CNN_U_Net_Baseline")

def build_vgg16_baseline():
    # Encoder VGG16 dipakai bersama; weights=None menghindari unduhan bobot eksternal dan membuat kedua baseline sama-sama belajar dari awal.
    base = VGG16(weights=None, include_top=False, input_shape=(*IMG_SIZE, 3))
    encoder = Model(base.input, [
        base.get_layer("block1_pool").output,  # 128x128
        base.get_layer("block2_pool").output,  # 64x64
        base.get_layer("block3_pool").output,  # 32x32
        base.get_layer("block4_pool").output,  # 16x16
        base.get_layer("block5_pool").output,  # 8x8
    ], name="shared_vgg16_encoder")
    encoder.trainable = False  # transfer-learning baseline; decoder dilatih
    inputs = Input((*IMG_SIZE, 6), name="pre_post_rgb")
    pre = Lambda(lambda z: preprocess_input(z * 255.0), name="vgg_preprocess_pre")(inputs[..., :3])
    post = Lambda(lambda z: preprocess_input(z * 255.0), name="vgg_preprocess_post")(inputs[..., 3:])
    pre_features = encoder(pre)
    post_features = encoder(post)
    # Fitur post dipakai untuk skip; fitur pre dan post digabung di bridge.
    skips = [concatenate([a, b]) for a, b in zip(pre_features[:4], post_features[:4])]
    bridge = concatenate([pre_features[4], post_features[4]])
    x = Conv2D(256, 1, padding="same", activation="relu")(bridge)
    outputs = decoder(x, skips)
    return Model(inputs, outputs, name="VGG16_U_Net_Baseline")

In [5]:
def train_and_evaluate(model, model_tag):
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss=bce_dice_loss, metrics=[dice_coef])
    checkpoint = os.path.join(OUTPUT_DIR, f"{model_tag}_best.keras")
    callbacks = [
        ModelCheckpoint(checkpoint, monitor="val_dice_coef", mode="max", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
        EarlyStopping(monitor="val_dice_coef", mode="max", patience=5, restore_best_weights=True, verbose=1),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
    model.save(os.path.join(OUTPUT_DIR, f"{model_tag}_final.keras"))
    return model, history

cnn_model = build_cnn_baseline()
cnn_model.summary()

Model: "CNN_U_Net_Baseline"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 pre_post_rgb (InputLayer)      [(None, 256, 256, 6  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 256, 256, 16  880         ['pre_post_rgb[0][0]']           
                                )                                                                 
                                                                                                  
 batch_normalization (BatchNorm  (None, 256, 256, 16  64         ['conv2d[0][0]']                 
 alization)                     )                                                

In [6]:
cnn_model, cnn_history = train_and_evaluate(cnn_model, "cnn_baseline")  

Epoch 1/15
980/980 [==============================] - ETA: 0s - loss: 1.4564 - dice_coef: 0.1407
Epoch 1: val_dice_coef improved from -inf to 0.16342, saving model to comparison_baselines\cnn_baseline_best.keras
980/980 [==============================] - 476s 483ms/step - loss: 1.4564 - dice_coef: 0.1407 - val_loss: 1.2604 - val_dice_coef: 0.1634 - lr: 1.0000e-04
Epoch 2/15
980/980 [==============================] - ETA: 0s - loss: 1.2011 - dice_coef: 0.1866
Epoch 2: val_dice_coef improved from 0.16342 to 0.20846, saving model to comparison_baselines\cnn_baseline_best.keras
980/980 [==============================] - 500s 510ms/step - loss: 1.2011 - dice_coef: 0.1866 - val_loss: 1.0592 - val_dice_coef: 0.2085 - lr: 1.0000e-04
Epoch 3/15
980/980 [==============================] - ETA: 0s - loss: 0.9845 - dice_coef: 0.2523
Epoch 3: val_dice_coef improved from 0.20846 to 0.26429, saving model to comparison_baselines\cnn_baseline_best.keras
980/980 [==============================] - 480s 48

In [7]:
vgg16_model = build_vgg16_baseline()
vgg16_model.summary()

Model: "VGG16_U_Net_Baseline"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 pre_post_rgb (InputLayer)      [(None, 256, 256, 6  0           []                               
                                )]                                                                
                                                                                                  
 tf.__operators__.getitem (Slic  (None, 256, 256, 3)  0          ['pre_post_rgb[0][0]']           
 ingOpLambda)                                                                                     
                                                                                                  
 tf.__operators__.getitem_1 (Sl  (None, 256, 256, 3)  0          ['pre_post_rgb[0][0]']           
 icingOpLambda)                                                                

In [8]:
vgg16_model, vgg16_history = train_and_evaluate(vgg16_model, "vgg16_baseline")

Epoch 1/15
980/980 [==============================] - ETA: 0s - loss: 1.4378 - dice_coef: 0.1423
Epoch 1: val_dice_coef improved from -inf to 0.16125, saving model to comparison_baselines\vgg16_baseline_best.keras
980/980 [==============================] - 569s 579ms/step - loss: 1.4378 - dice_coef: 0.1423 - val_loss: 1.2186 - val_dice_coef: 0.1612 - lr: 1.0000e-04
Epoch 2/15
980/980 [==============================] - ETA: 0s - loss: 1.1546 - dice_coef: 0.1979
Epoch 2: val_dice_coef improved from 0.16125 to 0.20612, saving model to comparison_baselines\vgg16_baseline_best.keras
980/980 [==============================] - 569s 580ms/step - loss: 1.1546 - dice_coef: 0.1979 - val_loss: 1.0772 - val_dice_coef: 0.2061 - lr: 1.0000e-04
Epoch 3/15
980/980 [==============================] - ETA: 0s - loss: 0.9647 - dice_coef: 0.2569
Epoch 3: val_dice_coef improved from 0.20612 to 0.27541, saving model to comparison_baselines\vgg16_baseline_best.keras
980/980 [==============================] - 5

In [13]:
def collect_predictions(model, dataset):
    y_true, y_prob = [], []
    for images, masks in dataset:
        y_true.append(masks.numpy())
        y_prob.append(model.predict(images, verbose=0))
    return np.concatenate(y_true, axis=0) >= 0.5, np.concatenate(y_prob, axis=0)

def select_threshold(model, dataset, candidates=np.arange(0.10, 0.91, 0.05)):
    # Threshold hanya dipilih dari validation set, dengan macro Dice per citra.
    y_true, y_prob = collect_predictions(model, dataset)
    best_threshold, best_dice = 0.5, -1.0
    for threshold in candidates:
        y_pred = y_prob >= threshold
        intersection = np.logical_and(y_true, y_pred).sum(axis=(1, 2, 3))
        true_sum = y_true.sum(axis=(1, 2, 3))
        pred_sum = y_pred.sum(axis=(1, 2, 3))
        dice = (2 * intersection + 1e-7) / (true_sum + pred_sum + 1e-7)
        mean_dice = float(dice.mean())
        if mean_dice > best_dice:
            best_threshold, best_dice = float(threshold), mean_dice
    return best_threshold, best_dice

def evaluate_test(model, dataset, threshold):
    y_true, y_prob = collect_predictions(model, dataset)
    y_pred = y_prob >= threshold
    intersection = np.logical_and(y_true, y_pred).sum(axis=(1, 2, 3))
    union = np.logical_or(y_true, y_pred).sum(axis=(1, 2, 3))
    true_sum = y_true.sum(axis=(1, 2, 3))
    pred_sum = y_pred.sum(axis=(1, 2, 3))
    dice = (2 * intersection + 1e-7) / (true_sum + pred_sum + 1e-7)
    iou = (intersection + 1e-7) / (union + 1e-7)
    return {"Dice_mean": float(dice.mean()), "Dice_std": float(dice.std()),
            "IoU_mean": float(iou.mean()), "IoU_std": float(iou.std()),
            "threshold": threshold}

models_to_compare = {
    "CNN baseline": cnn_model,
    "VGG16 baseline": vgg16_model,
    "ResNet50-U-Net": resnet50_unet,
}
thresholds = {}
validation_dice = {}
results = {}
for model_name, model in models_to_compare.items():
    thresholds[model_name], validation_dice[model_name] = select_threshold(model, val_ds)
    results[model_name] = evaluate_test(model, test_ds, thresholds[model_name])

print("Threshold terpilih dari validation set:", thresholds)
print("Validation macro Dice pada threshold terpilih:", validation_dice)
results


Threshold terpilih dari validation set: {'CNN baseline': 0.20000000000000004, 'VGG16 baseline': 0.5000000000000001, 'ResNet50-U-Net': 0.9000000000000002}
Validation macro Dice pada threshold terpilih: {'CNN baseline': 0.5875228873472726, 'VGG16 baseline': 0.5329017210445699, 'ResNet50-U-Net': 0.634654088528527}


{'CNN baseline': {'Dice_mean': 0.5933928306118195,
  'Dice_std': 0.2820217461094809,
  'IoU_mean': 0.475673425891379,
  'IoU_std': 0.27646280497030973,
  'threshold': 0.20000000000000004},
 'VGG16 baseline': {'Dice_mean': 0.537168970022879,
  'Dice_std': 0.2872481569791428,
  'IoU_mean': 0.4198632704334484,
  'IoU_std': 0.2761799105220463,
  'threshold': 0.5000000000000001},
 'ResNet50-U-Net': {'Dice_mean': 0.652093878872451,
  'Dice_std': 0.27433774247704484,
  'IoU_mean': 0.5396106842728785,
  'IoU_std': 0.2827476226300946,
  'threshold': 0.9000000000000002}}

In [14]:
import pandas as pd
comparison = pd.DataFrame(results).T
comparison

,Dice_mean,Dice_std,IoU_mean,IoU_std,threshold
CNN baseline,0.593393,0.282022,0.475673,0.276463,0.2
VGG16 baseline,0.537169,0.287248,0.419863,0.276180,0.5
ResNet50-U-Net,0.652094,0.274338,0.539611,0.282748,0.9


## Catatan perbandingan

- Dataset split dan pipeline preprocessing/augmentasi sama untuk kedua model; augmentasi hanya diterapkan pada train.
- CNN baseline belajar dari awal. VGG16 memakai bobot ImageNet yang dibekukan pada eksperimen baseline ini; decoder belajar dari data proyek.
- Untuk angka yang lebih stabil, ulangi eksperimen dengan beberapa seed dan laporkan rata-rata serta simpangan baku. Pertimbangkan juga precision/recall per piksel karena area kerusakan bisa jauh lebih kecil daripada area non-kerusakan.
- Hasil kedua model di sini belum dibandingkan dengan ResNet50-U-Net di notebook utama. Untuk tabel tiga model, evaluasi ResNet50 pada split dan protokol yang sama juga diperlukan.